In [18]:
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from sdv.single_table import CTGANSynthesizer  
from sdv.metadata import Metadata
import json
import numpy as np
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parents[2]
print(f"Project root: {project_root}")


Project root: /Users/fserracrespi/Desktop/PD_progression_ppmi


In [19]:
with open(project_root/"SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS_Domain_data.json", "r") as archivo:
    updrs_domain = json.load(archivo)

dominios_updrs = {
    'X_STATS': {

        'examen_motor': updrs_domain['SC_data'] + updrs_domain['UPDRS_MOTOR_STATS'],
        'impacto_motor': updrs_domain['SC_data'] + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS'],
        'non_motor': updrs_domain['SC_data'] + updrs_domain['UPDRS_NO_MOTOR_STATS']
    },

    'X_V06_STATS': {
        
        'examen_motor': updrs_domain['SC_data'] + updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_STATS'],
        'impacto_motor': updrs_domain['SC_data'] + updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_STATS'],
        'non_motor': updrs_domain['SC_data'] + updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_STATS']
    },

    'X_V06_DELTA': {
        'examen_motor': updrs_domain['SC_data'] + updrs_domain['UPDRS_MOTOR_V06'] + updrs_domain['UPDRS_MOTOR_delta'],
        'impacto_motor': updrs_domain['SC_data'] + updrs_domain['UPDRS_IMPACTO_MOTOR_V06'] + updrs_domain['UPDRS_IMPACTO_MOTOR_delta'],
        'non_motor': updrs_domain['SC_data'] + updrs_domain['UPDRS_NO_MOTOR_V06'] + updrs_domain['UPDRS_NO_MOTOR_delta']
    }
}

cols_updrs_full={}
for val in dominios_updrs:
    cols_updrs_val=[]
    for val_sub in dominios_updrs[val]:
        print(f"Evaluating domain: {val} - {val_sub}")
        cols_updrs_val+=dominios_updrs[val][val_sub]
    cols_updrs_full[val]=cols_updrs_val

cols_updrs_full['X_V06_STATS']=list(set(cols_updrs_full['X_V06_STATS']))
print(f"Number of unique features in X_V06_STATS: {len(cols_updrs_full['X_V06_STATS'])}")
cols_updrs_full['X_STATS']=list(set(cols_updrs_full['X_STATS']))
print(f"Number of unique features in X_STATS: {len(cols_updrs_full['X_STATS'])}")
cols_updrs_full['X_V06_DELTA']=list(set(cols_updrs_full['X_V06_DELTA']))
print(f"Number of unique features in X_V06_DELTA: {len(cols_updrs_full['X_V06_DELTA'])}")


Evaluating domain: X_STATS - examen_motor
Evaluating domain: X_STATS - impacto_motor
Evaluating domain: X_STATS - non_motor
Evaluating domain: X_V06_STATS - examen_motor
Evaluating domain: X_V06_STATS - impacto_motor
Evaluating domain: X_V06_STATS - non_motor
Evaluating domain: X_V06_DELTA - examen_motor
Evaluating domain: X_V06_DELTA - impacto_motor
Evaluating domain: X_V06_DELTA - non_motor
Number of unique features in X_V06_STATS: 360
Number of unique features in X_STATS: 301
Number of unique features in X_V06_DELTA: 242


# HY3 

In [20]:
HY3_v1=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3.csv', index_col=0)
HY3_v2=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_TEST.csv', index_col=0)
HY3_v3=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY3_TEST2.csv', index_col=0)

HY3_v2=HY3_v2[HY3_v2['STAGE_HY3']==2]
HY3_v3=HY3_v3[HY3_v3['STAGE_HY3']==2]

idx_HY3_v2=HY3_v2.index
idx_HY3_v3=HY3_v3.index

HY3_v2=HY3_v2.loc[idx_HY3_v2]
HY3_v3=HY3_v3.loc[idx_HY3_v3]

HY3_v2_final=pd.concat([HY3_v1, HY3_v2,HY3_v3], axis=0)


HY3_v2_final.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/y_HY3_v2_final.csv')
print(f"Final y_HY3 shape: {HY3_v2_final.shape}, class distribution:\n{HY3_v2_final['STAGE_HY3'].value_counts()}")

Final y_HY3 shape: (956, 1), class distribution:
STAGE_HY3
1    476
0    408
2     72
Name: count, dtype: int64


In [21]:
print(len(HY3_v1))
print(len(idx_HY3_v2))
print(len(idx_HY3_v3))

909
26
21


# MCID

In [22]:
MCID_v1=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID.csv', index_col=0)
MCID_v2=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_TEST.csv', index_col=0)
MCID_v3=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/UPDRS3/FULL_FEATURES/y_MCID_TEST2.csv', index_col=0)

MCID_v2=MCID_v2.loc[idx_HY3_v2]
MCID_v3=MCID_v3.loc[idx_HY3_v3]

MCID_v2_final=pd.concat([MCID_v1, MCID_v2,MCID_v3], axis=0)


MCID_v2_final.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/y_MCID_v2_final.csv')
print(f"Final y_HY3 shape: {MCID_v2_final.shape}, class distribution:\n{MCID_v2_final['STAGE_MCID'].value_counts()}")

Final y_HY3 shape: (956, 1), class distribution:
STAGE_MCID
1    498
2    280
0    178
Name: count, dtype: int64


# HY43

In [24]:
HY43_v1=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY43.csv', index_col=0)
HY43_v2=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY43_TEST.csv', index_col=0)
HY43_v3=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/y_HY43_TEST2.csv', index_col=0)

HY43_v2=HY43_v2[HY43_v2['STAGE_HY43']==1]
HY43_v3=HY43_v3[HY43_v3['STAGE_HY43']==1]

idx_HY43_v2=HY43_v2.index
idx_HY43_v3=HY43_v3.index

HY43_v2=HY43_v2.loc[idx_HY43_v2]
HY43_v3=HY43_v3.loc[idx_HY43_v3]


HY43_v2_final=pd.concat([HY43_v1, HY43_v2, HY43_v3], axis=0)

print(HY43_v2_final['STAGE_HY43'].value_counts())
HY43_v2_final.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/y_HY43_v2_final.csv')

STAGE_HY43
0    408
2    402
1    170
Name: count, dtype: int64


# X_STATS


## HY3


In [25]:
X_stats_v1=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS.csv', index_col=0)[cols_updrs_full['X_STATS']]
X_stats_v2=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS_TEST.csv', index_col=0)[cols_updrs_full['X_STATS']]
X_stats_v3=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS_TEST2.csv', index_col=0)[cols_updrs_full['X_STATS']]

X_stats_v2=X_stats_v2.loc[idx_HY3_v2]
X_stats_v2=X_stats_v2.loc[idx_HY3_v2]
X_stats_v3=X_stats_v3.loc[idx_HY3_v3]

X_stats_v2_final=pd.concat([X_stats_v1, X_stats_v2,X_stats_v3], axis=0)
X_stats_v2_final.shape
X_stats_v2_final.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_stats_v2_final_HY3.csv')

## HY43

In [26]:
X_stats_v1=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS.csv', index_col=0)[cols_updrs_full['X_STATS']]
X_stats_v2=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS_TEST.csv', index_col=0)[cols_updrs_full['X_STATS']]
X_stats_v3=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_STATS_TEST2.csv', index_col=0)[cols_updrs_full['X_STATS']]

X_stats_v2=X_stats_v2.loc[idx_HY43_v2]
X_stats_v3=X_stats_v3.loc[idx_HY43_v3]

X_stats_v2_final=pd.concat([X_stats_v1, X_stats_v2,X_stats_v3], axis=0)
X_stats_v2_final.shape
X_stats_v2_final.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_stats_v2_final_HY43.csv')

# X_VX_STATS

## HY3


In [27]:
X_v06_stats_v1=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+STATS.csv', index_col=0)
X_v08_stats_v2=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V08+STATS_TEST.csv', index_col=0)
X_v10_stats_v3=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V10+STATS_TEST2.csv', index_col=0)

X_v08_stats_v2=X_v08_stats_v2.loc[idx_HY3_v2]
X_v10_stats_v3=X_v10_stats_v3.loc[idx_HY3_v3]


cols_updrs_full['X_Vx_STATS'] = [
    col.replace('_V06', '_Vf').replace('_V08', '_Vf')
    for col in cols_updrs_full['X_V06_STATS']
]

X_v06_stats_v1.columns=[X.replace('_V06', '_Vf') for X in X_v06_stats_v1.columns]
X_v08_stats_v2.columns=[X.replace('_V08', '_Vf') for X in X_v08_stats_v2.columns]
X_v10_stats_v3.columns=[X.replace('_V10', '_Vf') for X in X_v10_stats_v3.columns]

X_vx_stats_v2_final=pd.concat([X_v06_stats_v1, X_v08_stats_v2,X_v10_stats_v3], axis=0)[cols_updrs_full['X_Vx_STATS']]
X_vx_stats_v2_final.shape
X_vx_stats_v2_final.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_stats_v2_final_HY3.csv')


## HY43

In [28]:
X_v06_stats_v1=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+STATS.csv', index_col=0)
X_v08_stats_v2=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V08+STATS_TEST.csv', index_col=0)
X_v10_stats_v3=pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V10+STATS_TEST2.csv', index_col=0)

X_v08_stats_v2=X_v08_stats_v2.loc[idx_HY43_v2]
X_v10_stats_v3=X_v10_stats_v3.loc[idx_HY43_v3]

cols_updrs_full['X_Vx_STATS'] = [
    col.replace('_V06', '_Vf').replace('_V08', '_Vf')
    for col in cols_updrs_full['X_V06_STATS']
]

X_v06_stats_v1.columns=[X.replace('_V06', '_Vf') for X in X_v06_stats_v1.columns]
X_v08_stats_v2.columns=[X.replace('_V08', '_Vf') for X in X_v08_stats_v2.columns]
X_v10_stats_v3.columns=[X.replace('_V10', '_Vf') for X in X_v10_stats_v3.columns]

X_vx_stats_v2_final=pd.concat([X_v06_stats_v1, X_v08_stats_v2,X_v10_stats_v3], axis=0)[cols_updrs_full['X_Vx_STATS']]
X_vx_stats_v2_final.shape
X_vx_stats_v2_final.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_stats_v2_final_HY43.csv')

# X_VX_DELTAS

## HY3


In [29]:
X_v06_delta_v1 = pd.read_csv(
    project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+Deltas.csv',
    index_col=0
)

X_v08_delta_v2 = pd.read_csv(
    project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V08+Deltas_TEST.csv',
    index_col=0
)

X_v10_delta_v3 = pd.read_csv(
    project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V10+Deltas_TEST2.csv',
    index_col=0
)


X_v08_delta_v2 = X_v08_delta_v2.loc[idx_HY3_v2]
X_v10_delta_v3 = X_v10_delta_v3.loc[idx_HY3_v3]

cols_updrs_full['X_Vx_DELTA'] = [
    col.replace('_V06', '_Vf').replace('_V04', '_Vm').replace('_BL', '_Vi')
    for col in cols_updrs_full['X_V06_DELTA']
]
# Rename columns for V06 dataset
X_v06_delta_v1.columns = [c.replace('_V06', '_Vf') for c in X_v06_delta_v1.columns]
X_v06_delta_v1.columns = [c.replace('_V04', '_Vm') for c in X_v06_delta_v1.columns]
X_v06_delta_v1.columns = [c.replace('_BL', '_Vi') for c in X_v06_delta_v1.columns]

# Rename columns for V08 dataset (FIXED)
X_v08_delta_v2.columns = [c.replace('_V08', '_Vf') for c in X_v08_delta_v2.columns]
X_v08_delta_v2.columns = [c.replace('_V06', '_Vm') for c in X_v08_delta_v2.columns]
X_v08_delta_v2.columns = [c.replace('_V04', '_Vi') for c in X_v08_delta_v2.columns]

# Rename columns for V08 dataset (FIXED)
X_v10_delta_v3.columns = [c.replace('_V10', '_Vf') for c in X_v10_delta_v3.columns]
X_v10_delta_v3.columns = [c.replace('_V08', '_Vm') for c in X_v10_delta_v3.columns]
X_v10_delta_v3.columns = [c.replace('_V06', '_Vi') for c in X_v10_delta_v3.columns]

X_vx_delta_v2_final = pd.concat([X_v06_delta_v1, X_v08_delta_v2,X_v10_delta_v3], axis=0)[cols_updrs_full['X_Vx_DELTA']]
X_vx_delta_v2_final.shape
X_vx_delta_v2_final.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_delta_v2_final_HY3.csv')

X_vx_delta_v2_final.shape

(956, 242)

## HY43

In [30]:
X_v06_delta_v1 = pd.read_csv(
    project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V06+Deltas.csv',
    index_col=0
)

X_v08_delta_v2 = pd.read_csv(
    project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V08+Deltas_TEST.csv',
    index_col=0
)

X_v10_delta_v3 = pd.read_csv(
    project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/HY_SCALE/FULL_FEATURES/X_V10+Deltas_TEST2.csv',
    index_col=0
)


X_v08_delta_v2 = X_v08_delta_v2.loc[idx_HY43_v2]
X_v10_delta_v3 = X_v10_delta_v3.loc[idx_HY43_v3]

cols_updrs_full['X_Vx_DELTA'] = [
    col.replace('_V06', '_Vf').replace('_V04', '_Vm').replace('_BL', '_Vi')
    for col in cols_updrs_full['X_V06_DELTA']
]
# Rename columns for V06 dataset
X_v06_delta_v1.columns = [c.replace('_V06', '_Vf') for c in X_v06_delta_v1.columns]
X_v06_delta_v1.columns = [c.replace('_V04', '_Vm') for c in X_v06_delta_v1.columns]
X_v06_delta_v1.columns = [c.replace('_BL', '_Vi') for c in X_v06_delta_v1.columns]

# Rename columns for V08 dataset (FIXED)
X_v08_delta_v2.columns = [c.replace('_V08', '_Vf') for c in X_v08_delta_v2.columns]
X_v08_delta_v2.columns = [c.replace('_V06', '_Vm') for c in X_v08_delta_v2.columns]
X_v08_delta_v2.columns = [c.replace('_V04', '_Vi') for c in X_v08_delta_v2.columns]

# Rename columns for V08 dataset (FIXED)
X_v10_delta_v3.columns = [c.replace('_V10', '_Vf') for c in X_v10_delta_v3.columns]
X_v10_delta_v3.columns = [c.replace('_V08', '_Vm') for c in X_v10_delta_v3.columns]
X_v10_delta_v3.columns = [c.replace('_V06', '_Vi') for c in X_v10_delta_v3.columns]

X_vx_delta_v2_final = pd.concat([X_v06_delta_v1, X_v08_delta_v2,X_v10_delta_v3], axis=0)[cols_updrs_full['X_Vx_DELTA']]
X_vx_delta_v2_final.shape
X_vx_delta_v2_final.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_delta_v2_final_HY43.csv')
print(X_vx_delta_v2_final.shape)

(980, 242)


# Uso de SMOTE y RandomOversampling

## X_STATS


In [31]:
X = pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_stats_v2_final_HY3.csv', index_col=0)
y = HY3_v2_final['STAGE_HY3'].copy()
X_class2 = X[y == 2]
X_class2.shape

(72, 301)

### CTGAN

In [32]:
metadata = Metadata.detect_from_dataframe(data=X_class2)

# Crear modelo
model = CTGANSynthesizer(metadata)

# 🔹 Crear modelo
model = CTGANSynthesizer(metadata)

# 🔹 Entrenar
model.fit(X_class2)

# 🔹 Generar nuevos datos
synthetic_data = model.sample(num_rows=30)

# 🔹 Unir dataset
data_final_IA = pd.concat([X, synthetic_data], ignore_index=True)
y_IA = pd.concat([y, pd.Series([2]*30)], ignore_index=True)
print("Original class distribution:")
print(y.value_counts()) 
print("\nNew class distribution after adding synthetic data:")
print(y_IA.value_counts())

data_final_IA.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_stats_v2_final_IA_HY3.csv')
y_IA.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/y_HY3_v2_Sampled_HY3.csv')

/Users/fserracrespi/Desktop/PD_progression_ppmi/.venv/lib/python3.10/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
NP3RTARL_min           2
NP2EAT_min             3
NP2SALV_var            11
NP1URIN_mean           11
NP3RTALL_min           3
NP3RTARL_var           11
NP3RIGN_min            4
NP2HOBB_std            11
NP3RIGLL_max           5
NP2SWAL_var            11
NP3POSTR_max           5
NP2TRMR_var            11
NP3PRSPL_var           11
NP3BRADY_max           4
NP2EAT_mean            11
NP3LGAGL_std           11
NP3KTRML_mean          11
NP2SALV_max            5
NP1DPRS_std            11
NP3SPCH_mean           11
NP3RTALJ_min           2
NP3PSTBL_min           4
NP1SLPD_min            4
NP3PTRML_mean          11
NP3RISNG_max           5
NP3RTARL_max           4
NP2WALK_min            4
NP1ANXS_var            11
NP1SLPD_std            11
NP1URIN_min            4
ENRLGBA                2
NP3GAIT_max     

### SMOTE

In [33]:
smote = SMOTE(
    sampling_strategy={2: 102},  # Aumentar la clase 2 a 100 muestras
    k_neighbors=3,
    random_state=42
)

X_SMOTE_res, y_train_res = smote.fit_resample(X, y)
print('Shapes after SMOTE resampling:')
print(X_SMOTE_res.shape, y_train_res.shape)
print('Class distribution after SMOTE resampling:')
print(pd.Series(y_train_res).value_counts())
X_SMOTE_res.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_stats_v2_final_SMOTE_HY3.csv')

Shapes after SMOTE resampling:
(986, 301) (986,)
Class distribution after SMOTE resampling:
STAGE_HY3
1    476
0    408
2    102
Name: count, dtype: int64


### RandomOversampling

In [34]:
ros = RandomOverSampler(
    sampling_strategy={2: 102},
    random_state=42)

X_ROS_res, y_ROS_res = ros.fit_resample(X, y)
print('Shapes after RandomOverSampler resampling:')
print(X_ROS_res.shape, y_ROS_res.shape)
print('Class distribution after RandomOverSampler resampling:')
print(pd.Series(y_ROS_res).value_counts())
X_ROS_res.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_stats_v2_final_ROS_HY3.csv')

Shapes after RandomOverSampler resampling:
(986, 301) (986,)
Class distribution after RandomOverSampler resampling:
STAGE_HY3
1    476
0    408
2    102
Name: count, dtype: int64


## X_Vx_STATS


In [35]:
X = pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_stats_v2_final_HY3.csv', index_col=0)
y = HY3_v2_final['STAGE_HY3'].copy()
X_class2 = X[y == 2]
X_class2.shape

(72, 360)

### CTGAN

In [36]:
metadata = Metadata.detect_from_dataframe(data=X_class2)

# Crear modelo
model = CTGANSynthesizer(metadata)

# 🔹 Crear modelo
model = CTGANSynthesizer(metadata)

# 🔹 Entrenar
model.fit(X_class2)

# 🔹 Generar nuevos datos
synthetic_data = model.sample(num_rows=30)

# 🔹 Unir dataset
data_final_IA = pd.concat([X, synthetic_data], ignore_index=True)
y_IA = pd.concat([y, pd.Series([2]*30)], ignore_index=True)
print("Original class distribution:")
print(y.value_counts()) 
print("\nNew class distribution after adding synthetic data:")
print(y_IA.value_counts())
data_final_IA.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_stats_v2_final_IA_HY3.csv')

/Users/fserracrespi/Desktop/PD_progression_ppmi/.venv/lib/python3.10/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
NP3RTARL_min           2
NP3FTAPL_Vf            5
NP3RTALL_min           3
NP2HOBB_std            11
NP2TRMR_var            11
NP3PRSPL_var           11
NP1SLPD_min            4
NP3KTRML_mean          11
NP3PSTBL_min           4
NP3PTRML_mean          11
NP3RISNG_max           4
NP2SPCH_Vf             4
NP3RTALU_std           11
NP1SLPD_var            11
NP1COG_mean            11
NP3PRSPR_var           11
NP2EAT_var             11
NP3FTAPL_var           11
NP3RISNG_var           11
NP3RTALL_std           11
NP3RISNG_Vf            5
NP1CNST_Vf             4
NP3TTAPR_mean          11
NP3RTALU_mean          11
NP3RTARU_max           4
NP3SPCH_min            3
NP1SLPN_max            5
NP2WALK_var            11
NP3RIGRL_std           11
NP1ANXS_mean           11
NP3RTALL_var           11
NP1LTHD_var 

### SMOTE

In [37]:
smote = SMOTE(
    sampling_strategy={2: 102},  # Aumentar la clase 2 a 100 muestras
    k_neighbors=3,
    random_state=42
)

X_SMOTE_res, y_train_res = smote.fit_resample(X, y)
print('Shapes after SMOTE resampling:')
print(X_SMOTE_res.shape, y_train_res.shape)
print('Class distribution after SMOTE resampling:')
print(pd.Series(y_train_res).value_counts())
X_SMOTE_res.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_stats_v2_final_SMOTE_HY3.csv')

Shapes after SMOTE resampling:
(986, 360) (986,)
Class distribution after SMOTE resampling:
STAGE_HY3
1    476
0    408
2    102
Name: count, dtype: int64


### RandomOversampling

In [38]:
ros = RandomOverSampler(
    sampling_strategy={2: 102},
    random_state=42)

X_ROS_res, y_ROS_res = ros.fit_resample(X, y)
print('Shapes after RandomOverSampler resampling:')
print(X_ROS_res.shape, y_ROS_res.shape)
print('Class distribution after RandomOverSampler resampling:')
print(pd.Series(y_ROS_res).value_counts())
X_ROS_res.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_stats_v2_final_ROS_HY3.csv')


Shapes after RandomOverSampler resampling:
(986, 360) (986,)
Class distribution after RandomOverSampler resampling:
STAGE_HY3
1    476
0    408
2    102
Name: count, dtype: int64


## X_Vx_Delta


In [39]:
X = pd.read_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_delta_v2_final_HY3.csv', index_col=0)
y = HY3_v2_final['STAGE_HY3'].copy()
X_class2 = X[y == 2]
X_class2.shape

(72, 242)

### CTGAN

In [40]:
metadata = Metadata.detect_from_dataframe(data=X_class2)

# Crear modelo
model = CTGANSynthesizer(metadata)

# 🔹 Crear modelo
model = CTGANSynthesizer(metadata)

# 🔹 Entrenar
model.fit(X_class2)

# 🔹 Generar nuevos datos
synthetic_data = model.sample(num_rows=30)

# 🔹 Unir dataset
data_final_IA = pd.concat([X, synthetic_data], ignore_index=True)
y_IA = pd.concat([y, pd.Series([2]*30)], ignore_index=True)
print("Original class distribution:")
print(y.value_counts()) 
print("\nNew class distribution after adding synthetic data:")
print(y_IA.value_counts())
data_final_IA.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_delta_v2_final_IA_HY3.csv')

/Users/fserracrespi/Desktop/PD_progression_ppmi/.venv/lib/python3.10/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
NP3RISNG_delta_Vf_Vi   11
NP3FTAPL_Vf            5
NP2DRES_delta_Vm_Vi    11
NP2TURN_delta_Vm_Vi    11
NP1CNST_delta_Vf_Vm    11
NP1LTHD_delta_Vf_Vi    11
NP2HOBB_delta_Vm_Vi    11
NP1HALL_delta_Vf_Vm    11
NP3BRADY_delta_Vf_Vi   11
NP3PRSPL_delta_Vm_Vi   11
NP3FRZGT_delta_Vm_Vi   11
NP1URIN_delta_Vm_Vi    11
NP2TRMR_Vf             4
NP2RISE_Vf             5
NP2SALV_delta_Vf_Vi    11
NP2HWRT_delta_Vf_Vi    11
NP1ANXS_delta_Vf_Vi    11
NP3RTALL_delta_Vf_Vi   11
NP2SALV_Vf             5
NP3BRADY_delta_Vf_Vm   11
NP3PTRML_delta_Vm_Vi   11
ENRLGBA                2
NP3RIGRL_delta_Vm_Vi   11
NP3TTAPL_delta_Vm_Vi   11
NP3RTARU_delta_Vm_Vi   11
NP2SPCH_Vf             4
NP3GAIT_delta_Vm_Vi    11
NP2HOBB_delta_Vf_Vi    11
NP2DRES_delta_Vf_Vi    11
NP1COG_Vf              4
NP1FATG_delta_Vf_Vi    11
NP3KTRM

### SMOTE

In [41]:
smote = SMOTE(
    sampling_strategy={2: 102},  # Aumentar la clase 2 a 100 muestras
    k_neighbors=3,
    random_state=42
)

X_SMOTE_res, y_train_res = smote.fit_resample(X, y)
print('Shapes after SMOTE resampling:')
print(X_SMOTE_res.shape, y_train_res.shape)
print('Class distribution after SMOTE resampling:')
print(pd.Series(y_train_res).value_counts())
X_SMOTE_res.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_delta_v2_final_SMOTE_HY3.csv')

Shapes after SMOTE resampling:
(986, 242) (986,)
Class distribution after SMOTE resampling:
STAGE_HY3
1    476
0    408
2    102
Name: count, dtype: int64


### RandomOversampling

In [42]:
ros = RandomOverSampler(
    sampling_strategy={2: 102},
    random_state=42)

X_ROS_res, y_ROS_res = ros.fit_resample(X, y)
print('Shapes after RandomOverSampler resampling:')
X_ROS_res.shape, y_ROS_res.shape
print('Class distribution after RandomOverSampler resampling:')
print(pd.Series(y_ROS_res).value_counts())
X_ROS_res.to_csv(project_root/'SCRIPTS/PYTHON/MODEL_ANALYSIS/DATA/V2_version/X_vx_delta_v2_final_ROS_HY3.csv')


Shapes after RandomOverSampler resampling:
Class distribution after RandomOverSampler resampling:
STAGE_HY3
1    476
0    408
2    102
Name: count, dtype: int64
